# CASE 02 Python EDA

2025 청년 순이동과 전 연령 시도 간 경로를 탐색한다. 원인은 단정하지 않는다.

In [1]:
from pathlib import Path
import sys

CASE_NAME = "02_youth_migration_dynamics"
RAW_NAME = "2025_domestic_migration_statistics.xlsx"
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "cases" / CASE_NAME,
]
CASE_DIR = next(
    (path.resolve() for path in candidates if (path / "data" / "raw" / RAW_NAME).exists()),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(
        f"{RAW_NAME}를 찾지 못했습니다. 저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )
sys.path.insert(0, str(CASE_DIR / "src"))

from constants import RAW_FILE_NAME
from data_preparation import load_and_prepare
from data_quality import run_quality_checks
from kpi_segmentation import run_kpi_segmentation
from parse_official_tables import verify_source_file
from statistical_analysis import run_statistical_analysis

RAW = CASE_DIR / "data" / "raw" / RAW_FILE_NAME
source = verify_source_file(RAW)
prepared = load_and_prepare(RAW)
tables = prepared.tables
print(source["sha256"])
print("stale sheet present:", tables.workbook["has_stale_monthly_sheet"])


FE066C40AAE0AE5C34C67947B404DC8943405C9BF0A7952DA09B0CF26D901E9D
stale sheet present: True


In [2]:
prepared.youth_profile[['sido','net_total','net_youth_20_39','net_20s','net_30s','typology_ko']]

,sido,net_total,net_youth_20_39,net_20s,net_30s,typology_ko
0,경기,32970,21053,7439,13614,청년 유입형
1,서울,-26769,17207,35937,-18730,초입 유입·후기 유출형
2,인천,32264,12472,5004,7468,청년 유입형
3,충북,10789,2543,450,2093,청년 유입형
4,대전,2824,2433,2248,185,청년 유입형
5,세종,-47,1245,294,951,청년 유입형
6,충남,8266,292,-1133,1425,후기 정착형
7,울산,-5474,-1112,-1154,42,후기 정착형
8,제주,-4273,-2446,-2198,-248,청년 유출형
9,강원,-1387,-4177,-3765,-412,청년 유출형


In [3]:
prepared.youth_mobility.tail()

,year,youth_movers,total_movers,total_mobility_rate,15-19,20-24,25-29,30-34,35-39,40-44,youth_share_of_movers
16,2021,3165198,7213422,14.052143,11.955787,21.719866,28.482865,25.554814,18.436004,13.892399,0.438793
17,2022,2759867,6152155,12.002062,10.860883,20.423884,25.394770,22.209943,15.750980,11.779241,0.448602
18,2023,2751786,6128738,11.982856,10.838147,20.207349,24.917029,23.242511,16.812992,12.132958,0.448997
19,2024,2826405,6283319,12.311241,11.309471,21.700087,25.711662,24.108936,17.519787,12.553631,0.449827
20,2025,2761243,6117784,12.010032,11.267803,22.369191,25.887511,23.377131,16.973712,12.212269,0.451347


In [4]:
prepared.capital_flows

,flow_block,movers,share
0,capital_to_capital,723872,0.331153
1,noncapital_to_noncapital,663504,0.303536
2,noncapital_to_capital,418501,0.191454
3,capital_to_noncapital,380036,0.173857


In [5]:
inter = prepared.inter_sido.sort_values('movers', ascending=False).head(10)
inter[['origin','destination','movers','flow_block']]

,origin,destination,movers,flow_block
128,서울,경기,276867,capital_to_capital
7,경기,서울,235668,capital_to_capital
55,경기,인천,73089,capital_to_capital
131,인천,경기,57580,capital_to_capital
48,서울,인천,46570,capital_to_capital
241,부산,경남,43163,noncapital_to_noncapital
30,경남,부산,40742,noncapital_to_noncapital
45,경북,대구,37837,noncapital_to_noncapital
184,경기,충남,36471,capital_to_noncapital
2,인천,서울,34098,capital_to_capital


서울은 20대 순유입·30대 순유출이 동시에 나타난다. 경기·인천은 20대와 30대가 모두 순유입이다. 시도 간 상위 경로는 수도권 내부이며, 이 OD는 전 연령이다.